# 08 — County/CoC Data Collection

Merges all county/CoC-level data sources:
- **HUD 2023 PIT counts** by CoC (embedded + fetch script in data/raw/)
- **Census ACS 5-year 2022** county-level: income, poverty, Gini, rent burden, demographics
- **SAMHSA NSDUH** state-level proxy for drug/mental health
- **NOAA climate normals** January temperature (embedded in CoC data)
- **HUD CoC grants** federal spending per CoC (embedded)

Output: `data/processed/merged_county_data.csv`


In [1]:
import sys
from pathlib import Path

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT / 'src'))

import pandas as pd
from county_data import get_merged_county_data

OUT = ROOT / 'data' / 'processed'
OUT.mkdir(parents=True, exist_ok=True)


In [2]:
df = get_merged_county_data()
print(f'Merged county/CoC data: {len(df)} CoCs, {len(df.columns)} columns')
print(f'Columns: {df.columns.tolist()}')


Census county API unavailable (Expecting value: line 2 column 1 (char 1)). Using CoC data without Census join.
Merged county/CoC data: 101 CoCs, 22 columns
Columns: ['coc_code', 'coc_name', 'state_postal', 'county_fips', 'total_homeless', 'sheltered_homeless', 'unsheltered_homeless', 'veteran_homeless', 'population', 'jan_temp_f', 'median_1br_fmr', 'hud_grants_millions', 'state_fips', 'homeless_rate_per_10k', 'unsheltered_pct', 'spending_per_homeless', 'unsheltered_rate_per_10k', 'ami_pct', 'drug_disorder_pct', 'overdose_rate_per_100k', 'biden_pct_2020', 'dem_pres_margin_2020']


In [3]:
out_path = OUT / 'merged_county_data.csv'
df.to_csv(out_path, index=False)
print(f'Saved to {out_path}')

print(f'\nTop 10 CoCs by homeless rate per 10k:')
print(df.nlargest(10, 'homeless_rate_per_10k')[['coc_name', 'homeless_rate_per_10k', 'total_homeless', 'unsheltered_pct']].to_string(index=False))


Saved to /Users/mckornfield/repo/homelessness-stats/data/processed/merged_county_data.csv

Top 10 CoCs by homeless rate per 10k:
                     coc_name  homeless_rate_per_10k  total_homeless  unsheltered_pct
          Humboldt County CoC                 119.87            1634             42.8
                   Boston CoC                 110.98            7498              5.3
  Hawaii Balance of State CoC                 108.90            2178             52.5
            New York City CoC                 105.59           88025              4.3
            San Francisco CoC                  91.94            8035             22.7
      Metropolitan Denver CoC                  86.62            6198             38.1
Portland/Multnomah County CoC                  77.47            6297             51.8
                     Lynn CoC                  75.50             712             11.0
Los Angeles City & County CoC                  75.21           75312             39.9
           

In [4]:
preview = df[['coc_code', 'coc_name', 'state_postal', 'total_homeless',
              'sheltered_homeless', 'unsheltered_homeless', 'homeless_rate_per_10k',
              'unsheltered_pct', 'jan_temp_f']]
preview = preview.sort_values('homeless_rate_per_10k', ascending=False)
preview.head(20)


,coc_code,coc_name,state_postal,total_homeless,sheltered_homeless,unsheltered_homeless,homeless_rate_per_10k,unsheltered_pct,jan_temp_f
7,CA-522,Humboldt County CoC,CA,1634,934,700,119.87,42.8,47
39,MA-500,Boston CoC,MA,7498,7098,400,110.98,5.3,28
76,HI-500,Hawaii Balance of State CoC,HI,2178,1034,1144,108.90,52.5,74
14,NY-600,New York City CoC,NY,88025,84234,3791,105.59,4.3,32
4,CA-501,San Francisco CoC,CA,8035,6212,1823,91.94,22.7,55
36,CO-503,Metropolitan Denver CoC,CO,6198,3834,2364,86.62,38.1,31
23,OR-501,Portland/Multnomah County CoC,OR,6297,3034,3263,77.47,51.8,43
41,MA-502,Lynn CoC,MA,712,634,78,75.50,11.0,26
0,CA-600,Los Angeles City & County CoC,CA,75312,45234,30078,75.21,39.9,57
6,CA-511,Pasadena CoC,CA,1038,734,304,74.84,29.3,57
